In [ ]:
%matplotlib inline
import os
import sys
import numpy as np
import torch
import torch.nn as nn
from torch.autograd import Variable
from torch.utils.data import DataLoader
from torchvision.transforms import Compose
from tqdm import tqdm
import mlflow.pytorch
import imgaug.augmenters as iaa
from pprint import pprint
import pandas as pd
import cv2
import plotly.express as px
import mlflow
from sklearn.metrics import roc_auc_score

import matplotlib.pyplot as plt
%matplotlib inline

sys.path.append('../../../')

from src import MODELS_DIR, MLFLOW_TRACKING_URI, DATA_PATH
from src.data import TrainValTestSplitter, MURASubset
from src.data.transforms import *
from src.features.augmentation import Augmentation
from src.models.autoencoders import BottleneckAutoencoder, BaselineAutoencoder
from src.models.gans import DCGAN
from src.models.vaetorch import VAE
from src.models import BaselineAutoencoder
from src.features.pixelwise_loss import PixelwiseLoss
from src.models.autoencoders import MaskedMSELoss
from src.features.topk import TopK

## Top K Loss: Best BaselineAutoencoder Photoshop

#### Get data from MLFlow

In [ ]:
# Connect to mlflow
client = mlflow.tracking.MlflowClient(MLFLOW_TRACKING_URI)
client.list_experiments()
# get the path of the saved model from mlflow
run_id = '5ca7f67c33674926a00590752c877fe5'
experiment = client.get_experiment('1')
path = f'{experiment.artifact_location}/{run_id}/artifacts/BaselineAutoencoder.pth'
path

#### Initilize parameters and load model

In [ ]:
num_workers = 7
log_to_mlflow = False
device = "cpu"

# Mlflow parameters
run_params = {
    'batch_size': 32,
    'image_resolution': (512, 512),
    'num_epochs': 1000,
    'batch_normalisation': True,
    'pipeline': {
        'hist_equalisation': True,
        'data_source': 'XR_HAND_PHOTOSHOP',
    },
    'masked_loss_on_val': True,
    'masked_loss_on_train': True,
    'soft_labels': True,
    'glr': 0.001,
    'dlr': 0.00005,
    'z_dim': 1000,
    'lr': 0.0001
}


# Preprocessing pipeline

composed_transforms_val = Compose([GrayScale(),
                                   HistEqualisation(active=run_params['pipeline']['hist_equalisation']),
                                   Resize(run_params['image_resolution'], keep_aspect_ratio=True),
                                   Augmentation(iaa.Sequential([iaa.PadToFixedSize(512, 512, position='center')])),
                                   # Padding(max_shape=run_params['image_resolution']),
                                   # max_shape - max size of image after augmentation
                                   MinMaxNormalization(),
                                   ToTensor()])

# get data

data_path = f'{DATA_PATH}/{run_params["pipeline"]["data_source"]}'
splitter = TrainValTestSplitter(path_to_data=data_path)

validation = MURASubset(filenames=splitter.data_val.path, true_labels=splitter.data_val.label,
                        patients=splitter.data_val.patient, transform=composed_transforms_val)
test = MURASubset(filenames=splitter.data_test.path, true_labels=splitter.data_test.label,
                  patients=splitter.data_test.patient, transform=composed_transforms_val)

val_loader = DataLoader(validation, batch_size=run_params['batch_size'], shuffle=True, num_workers=num_workers)
test_loader = DataLoader(test, batch_size=run_params['batch_size'], shuffle=True, num_workers=num_workers)

# get model (change path to path to a trained model

model = torch.load(path, map_location=lambda storage, loc: storage)

# set loss function

outer_loss = MaskedMSELoss(reduction='none')
model.eval().to(device)



### Validation

#### get pixelwise loss for validation

In [ ]:
evaluation = PixelwiseLoss(model=model, model_class='CAE', device=device, loss_function=outer_loss, masked_loss_on_val=True)
loss_dict = evaluation.get_loss(data = val_loader)

#### Get top k loss

In [ ]:
topk = TopK(loss=loss_dict['loss'], reduce_to_mean=True)
topk_sum = TopK(loss=loss_dict['loss'], reduce_to_mean=False)
label_list = [elem.item() for elem in loss_dict['label']]
topk_baseline_range = topk.get_range_topk_auc(start=1, end=20002, step=200, label=label_list)
topk_baseline_range_sum = topk_sum.get_range_topk_auc(start=1, end=10002, step=200, label=label_list)

In [ ]:
# transform dict to df
topk_baseline_range_df = pd.DataFrame.from_dict(topk_baseline_range)
topk_baseline_range_df_sum = pd.DataFrame.from_dict(topk_baseline_range_sum)

In [ ]:
# plot AUC vs K
fig = px.line(topk_baseline_range_df, x="K", y="AUC", title='AUC')
fig.show()

In [ ]:
# plot AUC vs K
fig = px.line(topk_baseline_range_df_sum, x="K", y="AUC", title='AUC')
fig.show()

### Test

#### get pixelwise loss for validation

In [ ]:
evaluation = PixelwiseLoss(model=model, model_class='CAE', device=device, loss_function=outer_loss, masked_loss_on_val=True)
loss_dict = evaluation.get_loss(data = test_loader)

#### Get top k loss

In [ ]:
topk = TopK(loss=loss_dict['loss'], reduce_to_mean=True)
label_list = [elem.item() for elem in loss_dict['label']]

In [ ]:
topk_baseline = topk.get_topk(k=8001)
roc_auc_score(label_list, topk_baseline)

In [ ]:
fig_loss_label = topk.get_pixelwise_plot(topk_baseline, label_list, bin_size=0.0002)
fig_loss_label.show()

## Top K Loss: Best BaselineAutoencoder

#### Get data from MLFlow

In [ ]:
# Connect to mlflow
client = mlflow.tracking.MlflowClient(MLFLOW_TRACKING_URI)
client.list_experiments()
# get the path of the saved model from mlflow
run_id = '0ca304cc755f4203a5c6576edb14e4b9'
experiment = client.get_experiment('1')
path = f'{experiment.artifact_location}/{run_id}/artifacts/baseline_autoencoder/data/model.pth'
path

#### Initilize parameters and load model

In [ ]:
num_workers = 7
log_to_mlflow = False
device = "cpu"

# Mlflow parameters
run_params = {
    'batch_size': 32,
    'image_resolution': (512, 512),
    'num_epochs': 1000,
    'batch_normalisation': True,
    'pipeline': {
        'hist_equalisation': True,
        'data_source': 'XR_HAND_CROPPED',
    },
    'masked_loss_on_val': True,
    'masked_loss_on_train': True,
    'soft_labels': True,
    'glr': 0.001,
    'dlr': 0.00005,
    'z_dim': 1000,
    'lr': 0.001
}


# Preprocessing pipeline

composed_transforms_val = Compose([GrayScale(),
                                   HistEqualisation(active=run_params['pipeline']['hist_equalisation']),
                                   Resize(run_params['image_resolution'], keep_aspect_ratio=True),
                                   Augmentation(iaa.Sequential([iaa.PadToFixedSize(512, 512, position='center')])),
                                   # Padding(max_shape=run_params['image_resolution']),
                                   # max_shape - max size of image after augmentation
                                   MinMaxNormalization(),
                                   ToTensor()])

# get data

data_path = f'{DATA_PATH}/{run_params["pipeline"]["data_source"]}'
splitter = TrainValTestSplitter(path_to_data=data_path)

validation = MURASubset(filenames=splitter.data_val.path, true_labels=splitter.data_val.label,
                        patients=splitter.data_val.patient, transform=composed_transforms_val)
test = MURASubset(filenames=splitter.data_test.path, true_labels=splitter.data_test.label,
                  patients=splitter.data_test.patient, transform=composed_transforms_val)

val_loader = DataLoader(validation, batch_size=run_params['batch_size'], shuffle=True, num_workers=num_workers)
test_loader = DataLoader(test, batch_size=run_params['batch_size'], shuffle=True, num_workers=num_workers)

# get model (change path to path to a trained model

model = torch.load(path, map_location=lambda storage, loc: storage)

# set loss function

outer_loss = MaskedMSELoss(reduction='none')
model.eval().to(device)



### Validation

#### get pixelwise loss for validation

In [ ]:
evaluation = PixelwiseLoss(model=model, model_class='CAE', device=device, loss_function=outer_loss, masked_loss_on_val=True)
loss_dict = evaluation.get_loss(data = val_loader)

#### Get top k loss

In [ ]:
topk = TopK(loss=loss_dict['loss'], reduce_to_mean=True)
topk_sum = TopK(loss=loss_dict['loss'], reduce_to_mean=False)
label_list = [elem.item() for elem in loss_dict['label']]
topk_baseline_range = topk.get_range_topk_auc(start=1, end=20002, step=200, label=label_list)
topk_baseline_range_sum = topk_sum.get_range_topk_auc(start=1, end=10002, step=200, label=label_list)

In [ ]:
# transform dict to df
topk_baseline_range_df = pd.DataFrame.from_dict(topk_baseline_range)
topk_baseline_range_df_sum = pd.DataFrame.from_dict(topk_baseline_range_sum)

In [ ]:
# plot AUC vs K
fig = px.line(topk_baseline_range_df, x="K", y="AUC", title='AUC')
fig.show()

In [ ]:
# plot AUC vs K
fig = px.line(topk_baseline_range_df_sum, x="K", y="AUC", title='AUC')
fig.show()

### Test

#### get pixelwise loss for validation

In [ ]:
evaluation = PixelwiseLoss(model=model, model_class='CAE', device=device, loss_function=outer_loss, masked_loss_on_val=True)
loss_dict = evaluation.get_loss(data = test_loader)

In [ ]:
topk = TopK(loss=loss_dict['loss'], reduce_to_mean=True)
topk_baseline = topk.get_topk(k=401)
label_list = [elem.item() for elem in loss_dict['label']]

In [ ]:
roc_auc_score(label_list, topk_baseline)

In [ ]:
fig_loss_label = topk.get_pixelwise_plot(topk_baseline, label_list, bin_size=0.0002)
fig_loss_label.show()